# 04 — Evaluation & Metrics Analysis

This notebook loads evaluation results and provides detailed
metric analysis, including:
- Per-track and per-source metric summaries
- Distribution plots (SDR, SI-SDR, SIR, SAR)
- Comparison between guitar 1 and guitar 2 performance
- Metric correlation analysis

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

## 1. Load evaluation results

Run the evaluation pipeline first:
```bash
python scripts/run_pipeline.py --config configs/unconditioned.yaml --checkpoint path/to/best.pt
```

In [ ]:
RUN_NAME = "unconditioned"
CHECKPOINT_NAME = "best"

metrics_dir = REPO_ROOT / "outputs" / "metrics" / RUN_NAME / CHECKPOINT_NAME
per_track_path = metrics_dir / "per_track_metrics.json"
summary_path = metrics_dir / "summary.json"

if per_track_path.exists():
    with open(per_track_path) as f:
        per_track = json.load(f)
    with open(summary_path) as f:
        summary = json.load(f)
    print("Loaded evaluation results.")
    print(f"Sources: {list(per_track.keys())}")
    for source, metrics in summary.items():
        if source.startswith("_"):
            continue
        print(f"\n{source}:")
        for k, v in metrics.items():
            if k != "num_tracks":
                print(f"  {k}: {v:.3f}" if v is not None else f"  {k}: N/A")
else:
    print(f"No results found at {metrics_dir}")
    print("Run the evaluation pipeline first, then re-run this cell.")
    per_track = None

## 2. Build analysis DataFrame

In [ ]:
if per_track is not None:
    rows = []
    for source, tracks in per_track.items():
        for track_name, metrics in tracks.items():
            for metric_name, values in metrics.items():
                finite_vals = [v for v in values if np.isfinite(v)]
                if finite_vals:
                    rows.append({
                        "source": source,
                        "track": track_name,
                        "metric": metric_name,
                        "median": np.median(finite_vals),
                        "mean": np.mean(finite_vals),
                        "std": np.std(finite_vals),
                        "n_windows": len(finite_vals),
                    })
    results_df = pd.DataFrame(rows)
    display(results_df.head(10))
else:
    results_df = None

## 3. Per-source metric distributions

In [ ]:
if results_df is not None:
    metric_names = ["SDR", "SI-SDR", "SIR", "SAR"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for ax, metric in zip(axes.flat, metric_names):
        subset = results_df[results_df["metric"] == metric]
        for source in subset["source"].unique():
            vals = subset[subset["source"] == source]["median"].values
            ax.hist(vals, bins=15, alpha=0.5, label=source, edgecolor="black")
        ax.set_xlabel(f"{metric} (dB)")
        ax.set_ylabel("Count")
        ax.set_title(f"{metric} distribution (per-track median)")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 4. Guitar 1 vs Guitar 2 comparison

In [ ]:
if results_df is not None:
    pivot = results_df.pivot_table(
        index=["track", "metric"], columns="source", values="median"
    ).reset_index()

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, metric in zip(axes, ["SDR", "SI-SDR", "SIR", "SAR"]):
        sub = pivot[pivot["metric"] == metric]
        if "guitar1" in sub.columns and "guitar2" in sub.columns:
            ax.scatter(sub["guitar1"], sub["guitar2"], alpha=0.7, edgecolors="black")
            lims = [min(sub[["guitar1", "guitar2"]].min()), max(sub[["guitar1", "guitar2"]].max())]
            ax.plot(lims, lims, "--", color="gray", alpha=0.5)
            ax.set_xlabel("Guitar 1")
            ax.set_ylabel("Guitar 2")
            ax.set_title(metric)
            ax.grid(True, alpha=0.3)
            ax.set_aspect("equal")

    plt.suptitle("Guitar 1 vs Guitar 2 metrics (per-track median)", fontsize=13)
    plt.tight_layout()
    plt.show()

## 5. Per-track breakdown

In [ ]:
if results_df is not None:
    sdr_df = results_df[results_df["metric"] == "SDR"].copy()
    sdr_pivot = sdr_df.pivot_table(index="track", columns="source", values="median")

    fig, ax = plt.subplots(figsize=(14, max(4, len(sdr_pivot) * 0.4)))
    sdr_pivot.plot.barh(ax=ax, edgecolor="black", alpha=0.7)
    ax.set_xlabel("SDR (dB)")
    ax.set_title("SDR per track")
    ax.grid(True, alpha=0.3, axis="x")
    ax.legend(title="Source")
    plt.tight_layout()
    plt.show()

## 6. Summary table

In [ ]:
if results_df is not None:
    summary_table = results_df.groupby(["source", "metric"]).agg(
        median_of_medians=("median", "median"),
        mean_of_medians=("median", "mean"),
        std_of_medians=("median", "std"),
    ).round(3)
    display(summary_table)